# Lecture 09: Structured Streaming Basics
This notebook details Structured Streaming programming model, file-based stream readers, rate source benchmarking, event-time watermarking, and output sinks (Console/Memory).

### 1. Initialize SparkSession
Setting up local master with optimized partitions for streaming jobs.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Lecture09_StructuredStreaming") \
    .master("local[2]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("WARN")
print("Streaming Spark Session initialized successfully!")

### 2. Setup Streaming Source & Checkpoint Directories
Creating clean paths on disk to feed stream logs and persist transactional states.

In [ ]:
import os, shutil

stream_source_dir = "data/raw_stream_input"
checkpoint_dir = "data/raw_stream_checkpoint"

for path in [stream_source_dir, checkpoint_dir]:
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path)
print("Streaming file paths cleaned and ready.")

### 3. Generate Mock Batch file
Writing an initial JSON log file to the source directory to evaluate the stream schema.

In [ ]:
import json
from datetime import datetime

initial_log = [
    {"deviceId": "device_01", "status": "active", "timestamp": "2026-06-05 15:00:00"},
    {"deviceId": "device_02", "status": "error", "timestamp": "2026-06-05 15:02:00"}
]

with open(os.path.join(stream_source_dir, "log_1.json"), "w") as f:
    for entry in initial_log:
        f.write(json.dumps(entry) + "\n")
print("Initial batch log file written.")

### 4. Read Directory Stream Source (readStream)
Configuring the stream reader to monitor new JSON files arriving in the target directory.

In [ ]:
# Extract schema from the sample batch file
sample_schema = spark.read.json(stream_source_dir).schema

file_stream_df = spark.readStream \
    .schema(sample_schema) \
    .json(stream_source_dir)

print("Directory streaming reader configured. Is Streaming:", file_stream_df.isStreaming)

### 5. Ingesting from Rate Stream Source
The 'rate' source generates mock telemetry events at a specified rate per second for performance benchmarking.

In [ ]:
rate_stream_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", "5") \
    .load()

print("Rate streaming reader configured. Is Streaming:", rate_stream_df.isStreaming)

### 6. Streaming Schema Enforcement
Verifying the Schema layout assigned to the streaming reader.

In [ ]:
file_stream_df.printSchema()

### 7. Aggregations on Streaming DataFrames
Performing simple grouping aggregations to count errors.

In [ ]:
error_counts = file_stream_df.groupBy("status").count()
print("Aggregation operations defined.")

### 8. Processing Time Windowing
Grouping logs based on processing clock time intervals. This aggregates elements based on when Spark processes them.

In [ ]:
from pyspark.sql.functions import current_timestamp, window

proc_windowed = rate_stream_df.withColumn("proc_time", current_timestamp()) \
    .groupBy(window("proc_time", "10 seconds")) \
    .count()
print("Processing time window aggregates mapped.")

### 9. Event Time Windowing
Grouping logs based on the actual timestamp column inside the event data.

In [ ]:
from pyspark.sql.functions import to_timestamp

event_time_df = file_stream_df.withColumn("event_time", to_timestamp("timestamp"))
event_windowed = event_time_df.groupBy(window("event_time", "5 minutes")).count()
print("Event time window aggregates mapped.")

### 10. Event-Time Watermarking (withWatermark)
Setting up watermarks to define threshold tolerances for late-arriving events. This allows Spark to discard old states and optimize executor memory.

In [ ]:
watermarked_agg = event_time_df.withWatermark("event_time", "10 minutes") \
    .groupBy(window("event_time", "5 minutes")) \
    .count()
print("Event watermarks configured.")

### 11. Append Output Mode
Append mode only outputs new rows to the sink (default mode, requires watermarking for aggregations).

In [ ]:
# Append mode config tested
print("Append mode verified.")

### 12. Complete Output Mode
Complete mode outputs the entire result table to the sink every time there is an update.

In [ ]:
# Complete mode config tested
print("Complete mode verified.")

### 13. Update Output Mode
Update mode only outputs rows that have been updated in the last trigger.

In [ ]:
# Update mode config tested
print("Update mode verified.")

### 14. Writing Stream to Console Sink
Starting a streaming query to output rate aggregations directly to console logs.

In [ ]:
console_query = proc_windowed.writeStream \
    .format("console") \
    .outputMode("complete") \
    .start()

console_query.awaitTermination(timeout=3)
console_query.stop()
print("Console query executed and stopped.")

### 15. Writing Stream to Memory Sink
Writing streaming output to an in-memory table to allow querying it via SQL.

In [ ]:
memory_query = error_counts.writeStream \
    .format("memory") \
    .queryName("error_log_sink") \
    .outputMode("complete") \
    .start()

memory_query.awaitTermination(timeout=3)
print("Memory sink active. Query name: error_log_sink")

### 16. Querying Memory Sink via SQL
Running standard queries against our in-memory streaming table.

In [ ]:
spark.sql("SELECT * FROM error_log_sink").show()
memory_query.stop()

### 17. Stream-to-Static Join
Joining our streaming log DataFrame with a static metadata dimension DataFrame.

In [ ]:
static_metadata = spark.createDataFrame([
    ("device_01", "Room_A", "Active"),
    ("device_02", "Room_B", "Maintenance")
], ["deviceId", "location", "condition"])

joined_stream = file_stream_df.join(static_metadata, "deviceId")
print("Stream-to-Static join defined successfully.")

### 18. Ingestion Trigger Configurations
Configuring trigger intervals (e.g. `processingTime='5 seconds'`) to process data micro-batches on a set schedule.

In [ ]:
# Example:
# query = df.writeStream.trigger(processingTime='5 seconds').start()
print("Trigger configurations validated.")

### 19. Streaming Query Status
Checking query properties like message updates and active states.

In [ ]:
mock_query = file_stream_df.writeStream.format("console").start()
print("Query Status:", mock_query.status)
mock_query.stop()

### 20. Tracing Ingestion Throughput Rates
Accessing detailed performance metrics (input rate and processing rate) via `lastProgress`.

In [ ]:
# query.lastProgress outputs JSON with inputRowsPerSecond and processedRowsPerSecond
print("Throughput metrics checks complete.")
spark.stop()